# 1. [calculate] labour productivity
- Save to `working_yearly` new table
**Assets**
- Total assets: book value of all assets
(i.e. intangible and tangible assets, stock, current and non-currents assets)#
- Total liabilities: sum of current liabilities (i.e. loans and short-term debt, creditors and non-current liabilities (i.e. long-term financial liabilities including borrowing from credit institutions and bonds issued).
- Leverage: ratio of total liabilities to total assets.
  
**Income**
- Operating revenue (turnover): sum of net sales, other operating revenues and stock variations.
- Wage bill: renumeration_employees
- Employment: number of employees on the company’s payroll. 
- Negative turnover values. Turnover is defined as the operating revenue in FAME. In a few cases, some companies report negative turnover values. We flag (but keep) those companies reporting negative turnover values.  
   
**Productivity** 
- GVA (Lars): wage bill + EBITDA
- GVA (bottom-up): profit_loss_pretax + interest_paid + depreciation + remuneration_employees
- Productivity: GVA / employees
- Average wage: wage bill / employees
- Use lns

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs

old_table_name = "fame_yearly_kp"
new_table_name = "working_yearly"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))

# Reference the existing deflated table
fame_yearly = con.table(old_table_name)

# Calculate the new metrics using Ibis lazy evaluation
# We use ibis.ifelse to safely handle natural logarithms of negative or zero GVA
working_yearly = fame_yearly.mutate(
    gva1 = fame_yearly.wages + fame_yearly.ebitda,
    gva2 =  fame_yearly.profit_loss_pretax +
            fame_yearly.interest_paid +
            fame_yearly.depreciation +
            fame_yearly.remuneration_employees
).mutate(
    gva1_per_worker = ibis._.gva1 / fame_yearly.employees,
    gva2_per_worker = ibis._.gva2 / fame_yearly.employees,
    average_wage = fame_yearly.wages / fame_yearly.employees
)

# Verify the final materialized table
table_t_working = ibis.memtable(working_yearly)
print(f"\nSample of {new_table_name}:")
display(table_t_working.sample(0.0001).execute())

❌ DATA_DIR path does not exist or is not a directory: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.07.30
✅ Inserting columns into new 'working_yearly' table: ('registered_number', 'year', 'employees', 'average_wage', 'gva1', 'gva2', 'gva1_per_worker', 'gva2_per_worker')
✅ Materialized 'working_yearly' table.
📊 Number of rows: 1,128,490
📊 Number of columns: 8

Head of working_yearly:


,registered_number,year,employees,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker
0,01104045,2006,136,34.133069,4854.740818,4529.091941,35.696624,33.302147
1,05300871,2006,36,14.396593,-404.605338,NaN,-11.239037,NaN
2,02842567,2006,14,47.505686,710.989526,903.578558,50.784966,64.541326
3,01693652,2006,12,29.560182,296.331643,285.447257,24.694304,23.787271
4,00150042,2006,4391,35.062981,231531.805878,228444.721360,52.728719,52.025671
...,...,...,...,...,...,...,...,...
203,02806113,2023,113,20.813138,2612.009668,NaN,23.115130,NaN
204,01104079,2023,64,33.167790,3123.534503,3325.260699,48.805227,51.957198
205,07505621,2024,413,23.881356,10833.000000,5112.000000,26.230024,12.377724
206,09618366,2024,453,18.633879,8880.506000,9148.268000,19.603766,20.194852


In [ ]:
working_yearly_skinny = table_t_working.select(
    "registered_number", "year",
    "employees", "average_wage", "gva1", "gva2",
    "gva1_per_worker", "gva2_per_worker"
)

print(f"✅ Inserting columns into new '{new_table_name}' table: {working_yearly_skinny.columns}")
con.create_table(new_table_name, working_yearly_skinny, overwrite=True)

# Verify the final materialized table
final_table = con.table(new_table_name)
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized '{new_table_name}' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print(f"\nHead of {new_table_name}:")
display(final_table.sample(200 / row_count).execute())

# [calculate] 2. TFP

$$\ln(Y_{it}) = \alpha_i + \gamma_t + \beta_K \ln(K_{it}) + \beta_L \ln(L_{it}) + \varepsilon_{it}$$
- $Y_{it}$: `gva1` or `gva2`
- $K_{it}$: `fixed_total`
- $L_{it}$: `employees`
Noting: `fixed_total` = `tangibles` + `intangibles` + `investments_other`, representing different types of capitals, and `total_assets` = `fixed_total` + `current_assets`, which we exclude as financial assets are non-productive.

In [ ]:
import ibis
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

# 3. Define the 4 model setups to iterate through
# Varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = [
    {'Y': 'gva1', 'K': 'fixed_total', 'L': 'employees', 'name': 'TFP_gva1_fixed'},
    {'Y': 'gva2', 'K': 'fixed_total', 'L': 'employees', 'name': 'TFP_gva2_fixed'}
]

# Dictionary to store the parameter outputs (\beta_K, \beta_L) and model summaries
parameter_tables = {}

for mod in models:
    y_col_name, k_col_name, l_col_name, name = mod['Y'], mod['K'], mod['L'], mod['name']

    # Validate columns: must be positive
    table_filtered: ibis.Table = table_t_working                    \
        .select([y_col_name, k_col_name, l_col_name])   \
        .filter(
            (table_t_working[y_col_name] > 0) &
            (table_t_working[k_col_name] > 0) &
            (table_t_working[l_col_name] > 0)
        )
    
    for col in table_filtered.columns:
        table_filtered = table_filtered.mutate(["ln_" + col]= np.log(table_filtered[col]))
    
    # Define Endogenous (Y) and Exogenous (X) variables
    Y = table_filtered['ln_Y'].execute()
    X = sm.add_constant(table_filtered[['ln_K', 'ln_L']].execute())
    
    # Estimate the model with Firm (\alpha_i) and Year (\gamma_t) Fixed Effects
    mod_ols = PanelOLS(Y, X, entity_effects=True, time_effects=True)
    
    # Fit model with firm-clustered standard errors
    res = mod_ols.fit(cov_type='clustered', cluster_entity=True)
    
    # Extract \beta_K and \beta_L
    beta_K = res.params['ln_K']
    beta_L = res.params['ln_L']
    
    # Calculate firm-year specific TFP (Solow Residual)
    # TFP_it = ln(Y_it) - \beta_K*ln(K_it) - \beta_L*ln(L_it)
    table_filtered[name] = table_filtered['ln_Y'] - (beta_K * table_filtered['ln_K']) - (beta_L * table_filtered['ln_L'])
    
    # Join the calculated TFP column back to the main dataframe
    table_t_working = table_t_working.join(valid_df[[name]], how='left')
    
    # Store the results and parameters
    parameter_tables[name] = {
        'beta_K': beta_K,
        'beta_L': beta_L,
        'gamma_t': res.estimated_effects.xs('time_effects', level=1), # Year fixed effects
        'summary': res.summary
    }

print("Panel regressions complete. 4 TFP variants added to the dataframe.")

NameError: name 'table_t_working' is not defined